The goal of this notebook is to get the time delay between the (bugged timestamps in the) kinect stream and the other streams in the .xdf file. 

# The problem

Due to a bug in LSL_Kinect, the timestamps in the kinect streams are relative to the computer's startup time, whereas timestamps should be defined by LSL to enable synchronization with other streams.  
As a consequence, the kinect streams are delayed compared to the other streams, and the value of the delay is unknown (but supposed constant).

*NOTE: the BUG has been corrected on 31 July 2023, for LSL_Kinect versions >= 1.2.0.*

# The solution
Hopefully, the kinect streams are also saved in .csv files, and the mouse streams are saved in other .csv files.

For each device, we have two streams: 
- the mocap stream, also saved in a .csv file 
- the marker stream, also saved in a .csv file 

Hence, we have :

- in the .xdf file : 
    - the kinect streams: with buggy timestamps (i.e., relative to the computer's startup time, which is unknown)
    - the mouse streams: with LSL timestamps (i.e., in sync with the other streams)

- in the .csv files :
    - the mouse streams, with timestamps as current time in milliseconds (e.g., 1616422003021 a [java currentTimeMillis](https://docs.oracle.com/javase/8/docs/api/java/lang/System.html#currentTimeMillis--))
    - the kinect streams, with timestamps as current time in milliseconds (e.g., '2021-03-22 15:06:43.021' a date-time string)

Stated differently, we have 3 time references:
- `LSL-kinect-time`: he bugged timestamps of the Kinect streams (relative to the computer's start-up time)
- `LSL-time`: the LSL timestamps (in sync with the other streams)
- `CSV-time`: the computer's current time in JAVA milliseconds (the time reference of all .csv files)

To know the relations between the 3 time references, one solution is to compute:
- **kinect-to-csv delay**: the delay between the kinect streams and the corresponding csv files
- **mouse-to-csv delay**: the delay between mouse streams and the corresponding csv files
- **kinect-to-mouse delay**: the delay between the kinect streams and the mouse streams

From the previous, we can compute the corrected kinect timestamps : the timestamps of the kinect stream shifted by the kinect-to-mouse delay


NOTE: if the kinect and mouse are registered on the same computer, the current time is the same for both csv files. If this is not the case, we would need to take into account the time difference between the two computers clocks (which is unknown, and is the reason we use LSL...). The good news is that ReArm registrations (normally) use the same computer for running LSL-mouse and LSL-kinect. 



## Computing the mouse-to-csv delay
The mouse-to-csv delay is the delay between the mouse streams and the mouse csv files.

As only the mouse markers csv files are systematically saved in the ReArm data set, we  will compare the timestamps from:  
- `LSL-time` in the mouse marker stream
- `CSV-time` in the mouse markers csv file

Do do so, we need to:
- read the the `LSL-time` from the mouse marker stream 
- select the corresponding csv file (if it exists)
- read the csv file to extract the `CSV-time` column
- check that we have the same number of rows in the csv file and in the LSL stream (it should be the case)
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay

## Computing the kinect-to-csv delay
The kinect-to-csv delay is the delay between the kinect streams and the kinect csv files.

Here, we take advantage of some very good news: the first column of the kinect mocap stream contains the CSV-time. Therefore, we can simply compare the timestamps of :  
- `LSL-kinect-time` in the kinect mocap stream 
- `CSV-time` in the first column of the kinect mocap stream 

Do do so, we merely need to:
- read the the `LSL-kinect-time` from the kinect mocap stream 
- read the first column of the kinect mocap stream to extract the `CSV-time` column
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay

## Computing the kinect-to-mouse delay
The kinect-to-mouse delay is the delay between the kinect streams and the mouse streams.

To do so, we simply go from the kinect to csv to mouse time references:  
- kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s

## Correcting the kinect timestamps
The corrected kinect timestamps are the timestamps of the kinect stream shifted by the kinect-to-mouse delay: 
- corrected_kinect_timestamp = LSL-kinect-time + kinect_to_mouse_delay_s




## The plan for the solution

In [ ]:
import pyxdf
import numpy as np
import tempfile
import os

# for the tests
import matplotlib.pyplot as plt

%matplotlib widget

In [ ]:
# this should be the set to false for production use
# doRunTests = False
# do_debug = False

# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True
    do_debug = True

if doRunTests:
    ##########################################################################################
    xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"
    ##########################################################################################

## Read the mouse and kinect streams from the .xdf file

In [ ]:
# read the xdf file
#   xdf_mouse_marker_time = mouse > marker > timestamp (correct LSL time, in seconds)
#   xdf_kinect_mocap_time = kinect > mocap > timestamp (buggy LSL time, in seconds)
#   csv_kinect_mocap_time = kinect > mocap > timeseries : col 1 (ir is the timestamp in 'currentTimeMillis')


def read_xdf_mouse_kinect(xdf_fullFname):

    # NOTE: do not forget to synchronize the clocks for all streams
    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        select_streams=[
            {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
            {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
            {"type": "Markers", "name": "Mouse"},
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    )

    kinect_mocap = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ][0]

    kinect_markers = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Markers-Kinect"
    ][0]

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
    ][0]

    xdf_mouse_marker_time = mouse_markers["time_stamps"]
    xdf_mouse_marker_data = mouse_markers["time_series"]
    xdf_kinect_mocap_time = kinect_mocap["time_stamps"]
    csv_kinect_mocap_time = kinect_mocap["time_series"][:, 0] / 1000.0

    return (
        xdf_mouse_marker_time,
        xdf_mouse_marker_data,
        xdf_kinect_mocap_time,
        csv_kinect_mocap_time,
    )


if doRunTests:

    def test_read_xdf_mouse_kinect():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        (
            xdf_mouse_marker_time,
            xdf_mouse_marker_data,
            xdf_kinect_mocap_time,
            csv_kinect_mocap_time,
        ) = read_xdf_mouse_kinect(xdf_fullFname)

        def print_stats(name, data):
            print(
                f"{name}: {data[0]:15.3f}s to {data[-1]:15.3f}s, {len(data):6.0f} samples for a duration of {data[-1] - data[0]:8.3f}s"
            )

        print_stats("LSL-time = xdf_mouse_marker_time", xdf_mouse_marker_time)
        print_stats("BUG-time = xdf_kinect_mocap_time", xdf_kinect_mocap_time)
        print_stats("CSV-time = csv_kinect_mocap_time", csv_kinect_mocap_time)

    test_read_xdf_mouse_kinect()

## Get the kinect-to-csv delay

In [ ]:
# get the kinect-to-csv delay
#   kinect_to_csv_delay_s =  xdf_kinect_mocap_time - csv_kinect_mocap_time / 1000


def get_kinect_to_csv_delay(xdf_fullFname):
    """get the time difference between the xdf and csv mocap time"""

    (
        xdf_mouse_marker_time,
        xdf_mouse_marker_data,
        xdf_kinect_mocap_time,
        csv_kinect_mocap_time,
    ) = read_xdf_mouse_kinect(xdf_fullFname)

    kinect_to_csv_delay = csv_kinect_mocap_time - xdf_kinect_mocap_time
    return kinect_to_csv_delay


if doRunTests:

    def test_get_kinect_to_csv_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

        print(f"kinect_to_csv_delay (mean): {np.mean(kinect_to_csv_delay):.3f} s")
        print(f"kinect_to_csv_delay  (std): {np.std(kinect_to_csv_delay):.6f} s")

        # plot the time difference
        plt.figure()
        t = np.arange(len(kinect_to_csv_delay))
        x = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
        plt.plot(t, x, ".", markersize=1)
        plt.title("Kinect to CSV time difference (zero centered)")
        plt.xlabel("Frame number")
        plt.ylabel("Time difference (s)")
        plt.grid()
        plt.show()

    test_get_kinect_to_csv_delay()

## Read one markers csv file

In [ ]:
# read the mouse marker csv file
#   csv_kinect_mocap_time = col 2 (it is the timestamp in 'currentTimeMillis')


def read_marker_csv_file(full_fname_marker_csv):
    """Read one marker csv file and return a list of [timestamp, marker] pairs

    Parameters
    ----------
    fnameMarkerCsv : str
        Full filename of the marker csv file

    Returns
    -------

    out : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    if not full_fname_marker_csv.endswith(".csv"):
        raise ValueError("The file must be a csv file")

    # NOTE: some mouse markers lack the quotes around the multiline markers
    # we need to add the quotes around the multiline markers before loading the file
    with open(full_fname_marker_csv, "r") as fname:
        txt = fname.readlines()

    lines = [line.split(",") for line in txt]

    # find the lines where token 3 is "\n" = start of a multiline marker
    for i in range(len(lines)):
        line = lines[i]
        # find the start of a multiline marker
        if len(line) == 3 and line[2] == "\n":
            # add a " before the end of the line
            txt[i] = txt[i][:-1] + '"\n'
            # find the end of the multiline marker
            for j in range(i + 1, len(lines)):
                # if we have a normal one-line-marker
                if len(lines[j]) == 3:
                    # add a " before the end of the line (of the previous line)
                    txt[j - 1] = txt[j - 1][:-1] + '"\n'
                    break
                # if are at the end of the file
                if j == len(lines) - 1 and len(lines[j]) != 3:
                    # add a " before the end of the line
                    txt[j] = txt[j][:-1] + '"\n'
                    break

    # write the modified file to a temporary file and load it with np.loadtxt
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as fname:
        fname.writelines(txt)
        tempFileName = fname.name

    lines = np.loadtxt(
        fname=tempFileName, skiprows=3, delimiter=",", quotechar='"', dtype=str
    )

    # remove the first column (we shall need only the timestamp and the marker to compare with the xdf file)
    lines = np.delete(lines, 0, 1)

    # return a list of [timestamp, marker] pairs
    out = []
    for line in lines:
        # WARNING : the timestamp must be in seconds (as in xdf files)
        timestamp = float(line[0]) / 1000
        marker = line[1]
        out.append([timestamp, [marker]])

    return out


def plot_markers_csv_time_difference(marker_list):
    """plot the time difference between the markers in the list"""
    plt.figure()
    t = np.arange(len(marker_list))
    x = [data[0] for data in marker_list]
    dx = np.diff(x)
    dx = np.insert(dx, 0, np.nan)

    plt.plot(t, dx, ".", markersize=10)
    plt.xlabel("Frame number")
    plt.ylabel("Time difference (s)")
    plt.grid()
    plt.show()

    print(f"From CSV: mouse Marker time difference. {len(marker_list)} markers")
    print("  DeltaT     Time        Marker")
    for i in range(len(marker_list)):
        print(f"{dx[i]:8.3f} {marker_list[i][0]:.3f} {marker_list[i][1]}")


if doRunTests:

    def test_readMarkerCsv():
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_p.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_p.csv"
        csv_marker_data = read_marker_csv_file(csv_fullFname)

        plot_markers_csv_time_difference(csv_marker_data)

    test_readMarkerCsv()

## Read all markers csv files in the folder

In [ ]:
# make a single list of markers from the multiple csv files for this xdf file
#   csv_marker_data = list of [timestamp, marker] pairs


def read_all_marker_csv_files(xdf_path):
    """Read all marker csv files in the visit path and return a list of [timestamp, marker]
    pairs

    Parameters
    ----------
    visit_path : str
        Full path of the visit directory

    Returns
    -------
    out : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    # get the list of all marker csv files in the visit path
    marker_files = [
        os.path.join(xdf_path, f)
        for f in os.listdir(xdf_path)
        if f.endswith(".csv") and "_l_m_" in f
    ]

    # read all the marker csv files
    marker_data_dict = {}
    for i in range(len(marker_files)):
        mouse_markers = read_marker_csv_file(marker_files[i])
        marker = {
            "data": mouse_markers,
            "start": mouse_markers[0][0],
            "path": marker_files[i],
        }
        marker_data_dict[i] = marker

    # sort the markers by their start time
    sorted_marker_data = sorted(marker_data_dict.items(), key=lambda x: x[1]["start"])

    # make a single list of markers from the multiple csv files for this xdf file
    all_marker_data = []
    for i in range(len(sorted_marker_data)):
        all_marker_data.extend(sorted_marker_data[i][1]["data"])

    return all_marker_data


if doRunTests:

    def test_readAllMarkerCsvs():
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/"
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle"
        all_marker_data = read_all_marker_csv_files(xdf_path)
        plot_markers_csv_time_difference(all_marker_data)

    test_readAllMarkerCsvs()

## Read the mouse and kinect streams from one .xdf file

In [ ]:
def read_xdf_mouse_markers(xdf_fullFname):
    # read the xdf file
    #   xdf_mouse_marker_time = mouse > marker > timestamp (correct LSL time, in seconds)
    #   xdf_mouse_marker_data = mouse > marker > data
    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        select_streams=[
            {"type": "Markers", "name": "Mouse"},
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    )

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
    ][0]

    xdf_mouse_marker_time = mouse_markers["time_stamps"]
    xdf_mouse_marker_data = mouse_markers["time_series"]

    return xdf_mouse_marker_time, xdf_mouse_marker_data


if doRunTests:

    def test_xdf_mouse_markers():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        # get the xdf marker data
        (
            xdf_mouse_marker_time,
            xdf_mouse_marker_data,
            xdf_kinect_mocap_time,
            csv_kinect_mocap_time,
        ) = read_xdf_mouse_kinect(xdf_fullFname)

        # plot the time difference
        plt.figure()
        t = np.arange(len(xdf_mouse_marker_time))
        x = xdf_mouse_marker_time
        dx = np.diff(x)
        dx = np.insert(dx, 0, np.nan)

        plt.plot(t, dx, ".", markersize=10)
        plt.title("From CSV: mouse Marker time difference")
        plt.xlabel("Frame number")
        plt.ylabel("Time difference (s)")
        plt.grid()
        plt.show()

        # add a leading nan to the first time difference
        print(
            f"From XDF: mouse Marker time difference. {len(xdf_mouse_marker_time)} markers"
        )
        for i in range(len(xdf_mouse_marker_time)):
            print(
                f"{dx[i]:8.3f} {xdf_mouse_marker_time[i]:.3f} {xdf_mouse_marker_data[i]}"
            )

    test_xdf_mouse_markers()

## Functions to compare lists of [timestamp, [marker] ]

In [ ]:
# list comparison functions


def define_shortest_longest_lists(list1, list2):
    """Define the shortest and the longest list"""

    shortest_list = list1
    longest_list = list2
    if len(list2) < len(list1):
        shortest_list = list2
        longest_list = list1

    return shortest_list, longest_list


def find_first_occurrence_of_short_list_in_long_list(shortest_list, longest_list):
    """Find the first occurrence of the shortest list in the longest list
    the lists are lists of markers, hence comparison is on list[i][1] (the marker)"""

    if do_debug:

        # create a debug directory if it does not exist
        if not os.path.exists("../debug"):
            os.makedirs("../debug")

        # save the shortest list to a file for debugging
        with open("../debug/shortest_list.txt", "w") as f:
            for item in shortest_list:
                marker = item[1]
                f.write(f"{marker}\n")

        # save the longest list to a file for debugging
        with open("../debug/longest_list.txt", "w") as f:
            for item in longest_list:
                marker = item[1]
                f.write(f"{marker}\n")

    i_beg = -1
    i_end = -1
    error_msg = ""
    for i in range(len(longest_list)):
        longest_list_marker = longest_list[i][1]
        shortest_list_marker = shortest_list[0][1]
        start_found = (longest_list_marker == shortest_list_marker) and i_beg == -1
        if start_found:
            i_beg = i
            nb_j = len(shortest_list)
            for j in range(len(shortest_list)):
                if longest_list[i + j][1] == shortest_list[j][1]:
                    i_end = i + j
                # if we are at the end of the list, we have found the end
                if j == len(shortest_list) - 1:
                    break

    if i_beg == -1:
        error_msg += "The shortest list is not found in the longest list\n"
    if i_end == -1:
        error_msg += "The end of the shortest list is not found in the longest list\n"

    return i_beg, i_end, error_msg


def get_timestamps_differences(longest_list, shortest_list, i_beg, i_end):
    """Get the differences between the timestamps of the common part and the shortest list"""

    if len(shortest_list) > len(longest_list):
        raise ValueError(
            "The shortest list (second argument) must be shorter than the longest list"
        )

    timestamps_common_part = [x[0] for x in longest_list[i_beg : i_end + 1]]
    timestamps_shortest_list = [x[0] for x in shortest_list]

    timestamps_differences = [
        timestamps_common_part[i] - timestamps_shortest_list[i]
        for i in range(len(shortest_list))
    ]

    # NOTE: we do not know whether csv or xdf is in the shortest list, hence we do not know the sign!
    # Too bad... BUT...
    # timestamps_differences MUST be positive values.
    # This is because csv time is UNIX time (seconds since 1970) and
    # xdf time is in seconds since the start of the recording (or something like that).

    timestamps_differences = [abs(x) for x in timestamps_differences]

    return timestamps_differences

## Compute the mouse-to-csv delay

In [ ]:
# get the mouse-to-csv delay
#   mouse_to_csv_delay_s = xdf_mouse_marker_time - csv_kinect_mocap_time / 1000


def get_mouse_to_csv_delay(xdf_mouse_marker_list, csv_mouse_marker_list):
    """Get the mouse-to-csv delay"""

    # find the shortest and the longest list
    shortest_list, longest_list = define_shortest_longest_lists(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )

    # find the first occurrence of the shortest list in the longest list
    i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
        shortest_list, longest_list
    )

    if i_beg == -1 or i_end == -1:
        raise ValueError(error_msg)

    # get the differences between the timestamps of the common part and the shortest list
    timestamps_differences = get_timestamps_differences(
        longest_list, shortest_list, i_beg, i_end
    )
    # as np array
    timestamps_differences = np.array(timestamps_differences)

    return timestamps_differences


if doRunTests:

    def test_get_mouse_to_csv_delay():

        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_path = os.path.dirname(xdf_fullFname)

        # get the csv marker data
        csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)
        csv_mouse_marker_time = [data[0] for data in csv_mouse_marker_list]
        csv_mouse_marker_data = [data[1] for data in csv_mouse_marker_list]

        # get the xdf marker data
        (
            xdf_mouse_marker_time,
            xdf_mouse_marker_data,
            xdf_kinect_mocap_time,
            csv_kinect_mocap_time,
        ) = read_xdf_mouse_kinect(xdf_fullFname)

        xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

        # get the mouse-to-csv delay
        mouse_to_csv_delay = get_mouse_to_csv_delay(
            xdf_mouse_marker_list, csv_mouse_marker_list
        )
        # as a np array
        mouse_to_csv_delay = np.array(mouse_to_csv_delay)

        print(f"mouse_to_csv_delay (median): {np.median(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay   (mean): {np.mean(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay    (std): {np.std(mouse_to_csv_delay):.6f} s")

        # plot the time difference
        plt.figure()
        t = np.arange(len(mouse_to_csv_delay))
        x = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)
        plt.plot(t, x, ".", markersize=10)
        # add a horizontal line at the mean - median value
        mean_minus_median = float(
            np.mean(mouse_to_csv_delay) - np.median(mouse_to_csv_delay)
        )
        plt.axhline(y=-mean_minus_median, color="r", linestyle="--", label="median")
        plt.axhline(y=0, color="b", linestyle="--", label="mean")
        plt.title("Mouse to CSV time difference (mean centered)")
        plt.xlabel("Frame number")
        plt.ylabel("Time difference (s)")
        # put the legend outside the plot
        plt.legend(loc="upper right", bbox_to_anchor=(1.5, 1))
        plt.grid()
        plt.show()

    test_get_mouse_to_csv_delay()

## Compute the kinect-to-mouse delay

In [ ]:
# get the kinect-to-mouse delay
#   kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s


def get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay):
    """Get the kinect-to-mouse delay"""

    kinect_to_cvs_delay_median = np.median(kinect_to_csv_delay)
    kinect_to_cvs_delay_mean = np.mean(kinect_to_csv_delay)
    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)

    mouse_to_csv_delay_median = np.median(mouse_to_csv_delay)
    mouse_to_csv_delay_mean = np.mean(mouse_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    # NOTE: the kinect to mouse delay is the difference of the means-medians
    # The mean & std is the simplest approach, if the distribution is normal
    # which should be true, as the error is due to random delays in the system
    kinect_to_mouse_delay_mean = kinect_to_cvs_delay_mean - mouse_to_csv_delay_mean

    kinect_to_mouse_delay_median = (
        kinect_to_cvs_delay_median - mouse_to_csv_delay_median
    )

    # NOTE: the variance of the difference is the sum of the variances **minus the covariance**
    # (e.g. https://en.wikipedia.org/wiki/Propagation_of_uncertainty)
    # If we assume that the two distributions are independent, the covariance is zero
    # hence computing the variance of the difference as the sum of the variances is correct.
    # If we assume that the two distributions are not independent, we should subtract the covariance,
    # but we do not have it. We still can compute the variance of the difference as the sum of the variances,
    # and this will be an **upper bound of the variance of the difference**.
    kinect_to_mouse_delay_std = np.sqrt(
        kinect_to_cvs_delay_std**2 + mouse_to_csv_delay_std**2
    )

    return (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    )


def display_kinect_to_mouse_delay(
    kinect_to_csv_delay,
    mouse_to_csv_delay,
):

    # boxplot the two distributions (mean centered)
    k_distrib = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
    m_distrib = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)

    plt.figure()
    plt.boxplot([k_distrib, m_distrib], showmeans=True)
    plt.title("Kinect to CSV delay vs Mouse to CSV delay")
    plt.xticks([1, 2], ["Kinect to CSV", "Mouse to CSV"])
    plt.ylabel("Time difference, mean centered (s)")
    plt.grid()
    # add the 95% confidence interval

    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    plt.errorbar(
        [1.2, 2.2],
        [0, 0],
        yerr=[1.96 * kinect_to_cvs_delay_std, 1.96 * mouse_to_csv_delay_std],
        fmt="o",
        color="b",
        label="95% confidence interval",
    )
    plt.legend()
    plt.show()


def get_kinect_to_mouse_time_delay(xdf_fullFname):
    xdf_path = os.path.dirname(xdf_fullFname)

    # get the csv marker data
    csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)

    # get the xdf marker data
    (
        xdf_mouse_marker_time,
        xdf_mouse_marker_data,
        xdf_kinect_mocap_time,
        csv_kinect_mocap_time,
    ) = read_xdf_mouse_kinect(xdf_fullFname)

    xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

    # get the mouse-to-csv delay
    mouse_to_csv_delay = get_mouse_to_csv_delay(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )

    # get the kinect-to-csv delay
    kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

    # get the kinect-to-mouse delay
    (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    ) = get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay)

    return (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
        kinect_to_csv_delay,
        mouse_to_csv_delay,
    )


if doRunTests:

    def test_get_kinect_to_mouse_time_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Reaching/ReArm_C1P02_20210409_V2_r.xdf"

        (
            kinect_to_mouse_delay_mean,
            kinect_to_mouse_delay_median,
            kinect_to_mouse_delay_std,
            kinect_to_csv_delay,
            mouse_to_csv_delay,
        ) = get_kinect_to_mouse_time_delay(xdf_fullFname)

        print(f"kinect_to_mouse_delay (median): {kinect_to_mouse_delay_median:.3f} s")
        print(f"kinect_to_mouse_delay   (mean): {kinect_to_mouse_delay_mean:.3f} s")
        print(f"kinect_to_mouse_delay    (std): {kinect_to_mouse_delay_std:.6f} s")

        # display the kinect to mouse delay
        display_kinect_to_mouse_delay(
            kinect_to_csv_delay,
            mouse_to_csv_delay,
        )

    test_get_kinect_to_mouse_time_delay()

In [ ]:
# correct the xdf_kinect_mocap_time
#   xdf_kinect_mocap_time_corrected = xdf_kinect_mocap_time - kinect_to_csv_delay_s

# test the corrected xdf_kinect_mocap_time
#   print the start-stop time of all streams in the xdf file, before and after correction
#      Expected: the corrected start-stop time should similar for all streams
#   plot the hand motion of the kinect mocap and mouse mocap (in the circular task case), before and after correction
#      Expected: the corrected kinect hand motion should be more synchronized with the mouse motion

# Test the solution on one file



In [ ]:
# test the corrected xdf_kinect_mocap_time
#   print the start-stop time of all streams in the xdf file, before and after correction
#      Expected: the corrected start-stop time should similar for all streams


## Utility functions for XDF files
def print_streams_names_type(xdf_fullFname):
    """Print the names and types of all streams in the xdf file"""

    xdf_data, header = pyxdf.load_xdf(filename=xdf_fullFname)

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, stream_type, stream_name):
    """Get the stream data from the xdf_data"""

    stream = [
        stream
        for stream in xdf_data
        if stream["info"]["type"][0] == stream_type
        and stream["info"]["name"][0] == stream_name
    ][0]

    return stream


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data


if doRunTests:

    def plot_t_xyz(t, x, y, z, name):
        plt.plot(t, x, ".", markersize=2, label=f"x.{name}")
        plt.plot(t, y, ".", markersize=2, label=f"y.{name}")
        plt.plot(t, z, ".", markersize=2, label=f"z.{name}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.grid()

    def test_correct_xdf_kinect_mocap_time():

        # NOTE: only circle task has a real record of the mouse motion
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"

        # NOTE: to get the exact name and type of the streams in the xdf file
        # print_streams_names_type(xdf_fullFname)

        # NOTE: only read what you need <=> super fast
        xdf_data, header = pyxdf.load_xdf(
            filename=xdf_fullFname,
            synchronize_clocks=True,
            select_streams=[
                {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
                {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
                {"type": "Markers", "name": "Mouse"},
                {"type": "MoCap", "name": "Mouse"},
                {"type": "Accelerometer", "name": "NIC-Accelerometer"},
            ],
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        )

        mouse_mocap = get_stream(xdf_data, "MoCap", "Mouse")
        kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
        acc_mocap = get_stream(xdf_data, "Accelerometer", "NIC-Accelerometer")

        mouse_markers = get_stream(xdf_data, "Markers", "Mouse")
        kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

        mouse_t = mouse_mocap["time_stamps"]
        mouse_x = mouse_mocap["time_series"][:, 0]
        mouse_y = mouse_mocap["time_series"][:, 1]
        mouse_z = mouse_mocap["time_series"][:, 2]  # not used

        acc_t = acc_mocap["time_stamps"]
        acc_x = acc_mocap["time_series"][:, 0]
        acc_y = acc_mocap["time_series"][:, 1]
        acc_z = acc_mocap["time_series"][:, 2]

        kinect_t = kinect_mocap["time_stamps"]
        WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
        WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
        WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

        (
            kinect_to_mouse_delay_mean,
            kinect_to_mouse_delay_median,
            kinect_to_mouse_delay_std,
            kinect_to_csv_delay,
            mouse_to_csv_delay,
        ) = get_kinect_to_mouse_time_delay(xdf_fullFname)

        # correct the xdf_kinect_mocap_time
        kinect_t_correct = kinect_t + kinect_to_mouse_delay_mean

        # plot the mouse
        plt.figure()
        plot_t_xyz(mouse_t, mouse_x, mouse_y, mouse_z, "Mouse")
        plt.show()

        # plot the accelerometer
        plt.figure()
        plot_t_xyz(acc_t, acc_x, acc_y, acc_z, "Accelerometer")
        plt.show()

        # plot the kinect
        plt.figure()
        plot_t_xyz(kinect_t, WristRight_X, WristRight_Y, WristRight_Z, "WristRight")
        plt.show()

        # plot the kinect x and y motion of the hand on the same plot
        plt.figure()
        plot_t_xyz(
            kinect_t_correct,
            WristRight_X,
            WristRight_Y,
            WristRight_Z,
            "WristRight_correct",
        )
        scale = 0.001  # define a compatible y scale for the mouse motion
        plot_t_xyz(mouse_t, mouse_x * scale, mouse_y * scale, mouse_z * scale, "Mouse")
        plt.show()

    test_correct_xdf_kinect_mocap_time()

# The code 

In [ ]:
# this should be the set to false for production use
# doRunTests = False
# do_debug = False

# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True
    do_debug = True

if doRunTests:
    ##########################################################################################
    xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"
    ##########################################################################################

## Get the list of the needed csv files for the Reaching and Circle task

We need the list of the .csv files corresponding to each .xdf file.  
We use the list given by `goodFiles.log` in the visit directory.  
Only the Reaching and Circle tasks contain an xdf file with a kinect stream.  

NOTE : for the mouse, we shall not use the data csv files, but only the marker csv files.

The csv files that are expected are: 

- the kinect csv files (NONE are mandatory) 
    - `*_k.csv`: kinect data file 
    - `*_k_m.csv`: the kinect marker file

- the mouse csv marker files for the Reaching task (at least ONE is mandatory) 
    - `*_r_l_m_mau_np.csv`: mouse marker file for maximal arm use with the non-paretic arm
    - `*_r_l_m_mau_p.csv`: mouse marker file for maximal arm use with the paretic arm
    - `*_r_l_m_sau_np.csv`: mouse marker file for spontaneous arm use with the non-paretic arm
    - `*_r_l_m_sau_p.csv`: mouse marker file for spontaneous arm use with the paretic arm

- the mouse csv marker files for the Circle task (at least ONE is mandatory) 
    - `*_c_l_m_np.csv`: mouse marker file for the non-paretic arm
    - `*_c_l_m_p.csv`: mouse marker file for the paretic arm


The minimal set of files that are mandatory **for each task** are:
- the xdf file (the file that contains the kinect stream with wrong timestamps + the mouse stream with correct timestamps) 
- one mouse marker csv file (the file that contains the mouse stream with correct timestamps)


## Create a list of [timestamp, marker] pairs from all mouse csv files corresponding to one xdf file

A single list of [timestamp, marker] pairs from all mouse csv files corresponding to one xdf file should correspond to the mouse marker stream in the xdf file.

In [ ]:
def get_mouse_marker_dict(xdf_csv):
    """Get the mouse markers from the csv files list and return them as a dictionary containing :
    - list of [timestamp, marker] pairs
    - list of csv file names"""

    mouse_marker_list = []
    csv_fname_list = []

    for csv_fname in xdf_csv:
        if "_l_m" in csv_fname:
            csv_fname_list.append(csv_fname)
            fnameMarkerCSV = os.path.join(xdf_path, csv_fname)
            lines = read_marker_csv_file(fnameMarkerCSV)
            mouse_marker_list.extend(lines)

    # sort the markers by timestamp
    mouse_marker_list.sort(key=lambda x: x[0])

    # create a dictionary with mouse_marker_list and csv_fname_list
    mouse_marker_dict = {}
    mouse_marker_dict["mouse_marker_list"] = mouse_marker_list
    mouse_marker_dict["csv_fname_list"] = csv_fname_list

    return mouse_marker_dict


def print_mouse_marker_dict(mouse_marker_dict):
    """Print the mouse marker dictionary"""

    print(f"\nCSV files ({len(mouse_marker_dict['csv_fname_list'])} files):")
    for csv in mouse_marker_dict["csv_fname_list"]:
        print(csv)

    print(f"\nMouse markers ({len(mouse_marker_dict['mouse_marker_list'])} markers):")
    for marker in mouse_marker_dict["mouse_marker_list"]:
        print(marker)

    return


# def get_mouse_marker_list_for_reach_and_circle(xdf_csv_files, mandatory_files_error):
#     """Get the mouse marker list for reaching and circle tasks"""

#     mouse_markers_r = []
#     mouse_markers_c = []

#     if not mandatory_files_error:
#         for xdf_csv in xdf_csv_files:
#             if xdf_csv["xdf"].endswith("_r.xdf"):
#                 mouse_markers_r = get_mouse_marker_dict(xdf_csv["csv"])
#             if xdf_csv["xdf"].endswith("_c.xdf"):
#                 mouse_markers_c = get_mouse_marker_dict(xdf_csv["csv"])

#     return mouse_markers_r, mouse_markers_c


if doRunTests:

    def test_get_mouse_marker_csv_dict(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)

        # print the errors FIRST (if any)
        # Remember: 1 xdf file corresponds to several csv files...
        for xdf_csv in xdf_and_csv_files_in_visit:
            xdf_file = xdf_csv["xdf"]
            csv_files = xdf_csv["csv"]
            mandatory_files_error = xdf_csv["mandatory_files_error"]
            expected_files_warning = xdf_csv["expected_files_warning"]

            print(f"\nXDF: {xdf_file}")
            if mandatory_files_error:
                print(f"  >>>>>> ERROR: {mandatory_files_error}")

        # print the mouse markers for the reaching and circle tasks
        for xdf_csv in xdf_and_csv_files_in_visit:
            print(f"\nXDF: {xdf_file}")
            if xdf_csv["xdf"].endswith("_r.xdf"):
                mouse_markers_r = get_mouse_marker_dict(csv_files)
                print_mouse_marker_dict(mouse_markers_r)

            if xdf_csv["xdf"].endswith("_c.xdf"):
                mouse_markers_c = get_mouse_marker_dict(csv_files)
                print_mouse_marker_dict(mouse_markers_c)

    test_get_mouse_marker_csv_dict(xdf_path)

## Read the xdf file and get the kinect markers and mouse markers streams



In [ ]:
import os
import pyxdf


def load_xdf_kinect_mouse_streams(xdf_fullFname):
    """Load the xdf file and return the kinect and mouse streams"""

    extension = os.path.splitext(xdf_fullFname)[1]
    if extension != ".xdf":
        return []  # empty list if the file is not an xdf file

    # NOTE: There is a bug in the kinect markers stream: the csv is not exactly the same...
    #       So we shall use the MoCap stream to get the delay (
    #       hopefully it is the same, and there is a kinect MoCap csv file for each xdf file

    # NOTE: do not forget to synchronize the clocks for all streams
    data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        select_streams=[
            {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
            {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
            {"type": "Markers", "name": "Mouse"},
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    )
    kinect_mocap = [
        stream for stream in data if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ][0]
    kinect_markers = [
        stream
        for stream in data
        if stream["info"]["name"][0] == "EuroMov-Markers-Kinect"
    ][0]
    mouse_markers = [stream for stream in data if stream["info"]["name"][0] == "Mouse"][
        0
    ]

    return kinect_mocap, kinect_markers, mouse_markers


def print_markers_stream(stream):
    """Print a markers stream"""

    n_lines = len(stream["time_series"])
    name = stream["info"]["name"][0]
    print(f"\n{name} ({n_lines} markers):")
    for i in range(n_lines):
        print(f"{stream['time_stamps'][i]:.3f}, {stream['time_series'][i]}")

    return


def marker_list(stream):
    """Return a list of markers from a markers stream"""

    markers_list = []
    n_lines = len(stream["time_series"])
    for i in range(n_lines):
        markers_list.append([stream["time_stamps"][i], stream["time_series"][i]])

    return markers_list


def get_all_csv_marker_of_type_list(csv_rel_fnames, marker_name_pattern):
    """Get a list of all markers from the csv files of marker_name_pattern type sorted by timestamp"""

    markers_csv_fname_list = [
        csv for csv in csv_rel_fnames if marker_name_pattern in csv
    ]

    markers_csv_list = []
    for markers_csv_fname in markers_csv_fname_list:
        markers_csv = read_marker_csv_file(os.path.join(xdf_path, markers_csv_fname))
        for i in range(len(markers_csv)):
            markers_csv_list.append(markers_csv[i])

    # sort markers_csv_list by timestamp (first column)
    markers_csv_list.sort(key=lambda x: x[0])

    return markers_csv_list


def get_marker_csv_all_list(csv_rel_fnames):
    """Get a list of all markers from the csv files sorted by timestamp"""

    markers_csv_fname_list = [
        csv for csv in csv_rel_fnames if "_k_m" in csv or "_l_m" in csv
    ]

    markers_csv_list = []
    for markers_csv_fname in markers_csv_fname_list:
        markers_csv = read_marker_csv_file(os.path.join(xdf_path, markers_csv_fname))
        for i in range(len(markers_csv)):
            markers_csv_list.append(markers_csv[i])

    # sort markers_csv_list by timestamp (first column)
    markers_csv_list.sort(key=lambda x: x[0])

    return markers_csv_list


def print_marker_csv_files_by_file(csv_rel_fnames, marker_name_pattern):
    """Print all marker csv files, file by file"""

    # print the corresponding markers csv files
    markers_csv_fname_list = [
        csv for csv in csv_rel_fnames if marker_name_pattern in csv
    ]

    for markers_csv_fname in markers_csv_fname_list:
        markers_csv = read_marker_csv_file(os.path.join(xdf_path, markers_csv_fname))

        n_lines = len(markers_csv)
        print(f"\n{markers_csv_fname} ({n_lines}):")
        for i in range(n_lines):
            print(f"{markers_csv[i][0]:.3f}, {markers_csv[i][1]}")

    return


def print_marker_list(markers_list, title=""):
    """Print a list of markers"""

    n_lines = len(markers_list)
    print(f"\n{title} ({len(markers_list)}):")
    for i in range(n_lines):
        print(f"{markers_list[i][0]:.3f}, {markers_list[i][1]}")

    return


if doRunTests:

    def test_get_marker_xdf_dict(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)

        # print the errors FIRST (if any)
        # Remember: 1 xdf file corresponds to several csv files...
        for xdf_csv in xdf_and_csv_files_in_visit:
            xdf_file = xdf_csv["xdf"]
            csv_files = xdf_csv["csv"]
            mandatory_files_error = xdf_csv["mandatory_files_error"]
            expected_files_warning = xdf_csv["expected_files_warning"]

            print(f"\nXDF: {xdf_file}")
            if mandatory_files_error:
                print(f"  >>>>>> ERROR: {mandatory_files_error}")

        for xdf_csv in xdf_and_csv_files_in_visit:
            xdf_file = xdf_csv["xdf"]
            csv_files = xdf_csv["csv"]
            mandatory_files_error = xdf_csv["mandatory_files_error"]
            expected_files_warning = xdf_csv["expected_files_warning"]

            print(
                "\n*****************************************************************************"
                + "\n*****************************************************************************"
            )
            print(f"\nXDF: {xdf_file}")

            kinect_mocap, kinect_markers, mouse_markers = load_xdf_kinect_mouse_streams(
                os.path.join(visit_path, xdf_file)
            )

            print_markers_stream(kinect_markers)
            print_markers_stream(mouse_markers)

    test_get_marker_xdf_dict(xdf_path)

## Utility functions to compare lists of [timestamp, marker] pairs

In [ ]:
def define_shortest_longest_lists(list1, list2):
    """Define the shortest and the longest list"""

    shortest_list = list1
    longest_list = list2
    if len(list2) < len(list1):
        shortest_list = list2
        longest_list = list1

    return shortest_list, longest_list


def find_first_occurrence_of_short_list_in_long_list(shortest_list, longest_list):
    """Find the first occurrence of the shortest list in the longest list
    the lists are lists of markers, hence comparison is on list[i][1] (the marker)"""

    if do_debug:

        # # change the working directory to the parent directory
        # if os.getcwd().endswith("notebooks"):
        #     os.chdir("..")

        # # create a debug directory if it does not exist
        # if not os.path.exists("debug"):
        #     os.makedirs("debug")

        # save the shortest list to a file for debugging
        with open("../debug/shortest_list.txt", "w") as f:
            for item in shortest_list:
                marker = item[1]
                f.write(f"{marker}\n")

        # save the longest list to a file for debugging
        with open("../debug/longest_list.txt", "w") as f:
            for item in longest_list:
                marker = item[1]
                f.write(f"{marker}\n")

    i_beg = -1
    i_end = -1
    error_msg = ""
    for i in range(len(longest_list)):
        longest_list_marker = longest_list[i][1]
        shortest_list_marker = shortest_list[0][1]
        start_found = (longest_list_marker == shortest_list_marker) and i_beg == -1
        if start_found:
            i_beg = i
            nb_j = len(shortest_list)
            for j in range(len(shortest_list)):
                if longest_list[i + j][1] == shortest_list[j][1]:
                    i_end = i + j
                # if we are at the end of the list, we have found the end
                if j == len(shortest_list) - 1:
                    break

    if i_beg == -1:
        error_msg += "The shortest list is not found in the longest list\n"
    if i_end == -1:
        error_msg += "The end of the shortest list is not found in the longest list\n"

    return i_beg, i_end, error_msg


def get_timestamps_differences(longest_list, shortest_list, i_beg, i_end):
    """Get the differences between the timestamps of the common part and the shortest list"""

    if len(shortest_list) > len(longest_list):
        raise ValueError(
            "The shortest list (second argument) must be shorter than the longest list"
        )

    timestamps_common_part = [x[0] for x in longest_list[i_beg : i_end + 1]]
    timestamps_shortest_list = [x[0] for x in shortest_list]

    timestamps_differences = [
        timestamps_common_part[i] - timestamps_shortest_list[i]
        for i in range(len(shortest_list))
    ]

    # NOTE: timestamps_differences MUST be positive values only.
    # This is because csv time is UNIX time (seconds since 1970) and
    # xdf time is in seconds since the start of the recording (or something like that).

    timestamps_differences = [abs(x) for x in timestamps_differences]

    return timestamps_differences

## A minimal class to manage the delay results for an xdf file

In [ ]:
class Delay_Results_for_xdf:
    def __init__(self, xdf_csv):
        self.xdf_csv = xdf_csv
        self.xdf_rel_fname = xdf_csv["xdf"]
        self.csv_rel_fnames = xdf_csv["csv"]
        self.mandatory_files_error = xdf_csv["mandatory_files_error"]
        self.expected_files_warning = xdf_csv["expected_files_warning"]
        self.results = {
            "error_msg": self.mandatory_files_error,
            "timestamps_differences": None,
            "iBeg": None,
            "iEnd": None,
            "longest_list": None,
            "shortest_list": None,
            "markers_csv_list": None,
            "markers_xdf_list": None,
            "from_marker": "",
        }

    def __str__(self):
        # print all the (printable) attributes of the object
        out = ""
        out += f"xdf_rel_fname:\n  {self.xdf_rel_fname}\n"
        out += f"csv_rel_fnames:\n  {self.csv_rel_fnames}\n"
        out += f"mandatory_files_error:\n  {self.mandatory_files_error}\n"
        out += f"expected_files_warning:\n  {self.expected_files_warning}\n"
        out += "return_dict: \n"
        for key, value in self.results.items():
            out += f"  {key}: {value}\n"
        return out

    def __are_results_set__(self):
        no_error = self.results["error_msg"] is None or self.results["error_msg"] == ""
        return all([x is not None for x in self.results.values()]) and no_error

    def summary(self):
        """Print the marker delays results summary"""

        print("")
        print(
            "*****************************************************************************"
        )
        print(f"XDF: {self.xdf_rel_fname}: ")

        if not self.__are_results_set__():
            if self.mandatory_files_error:
                print(f"  >>>>>> ERROR: {self.mandatory_files_error}")
            print("  The results are not set")
            return

        # get the values from the dictionary
        timestamps_differences = self.results["timestamps_differences"]
        i_beg = self.results["iBeg"]
        i_end = self.results["iEnd"]
        longest_list = self.results["longest_list"]
        shortest_list = self.results["shortest_list"]
        markers_csv_list = self.results["markers_csv_list"]
        markers_xdf_list = self.results["markers_xdf_list"]
        from_marker = self.results["from_marker"]

        # start printing
        print("")
        print_marker_list(
            markers_csv_list, f"Sorted {from_marker} markers from csv files"
        )
        print_marker_list(markers_xdf_list, f"{from_marker} markers from xdf")

        print("")
        print(f"First occurrence of the shortest list in the longest list: {i_beg}")
        print(f"Last occurrence of the shortest list in the longest list: {i_end}")

        # print the common part of the longest list
        common_part = longest_list[i_beg : i_end + 1]
        print_marker_list(common_part, "Common part of the longest list")

        # print the timestamps of the common part
        timestamps_common_part = [x[0] for x in common_part]
        print(f"\nTimestamps of the common part: {timestamps_common_part}")

        # print the timestamps of the shortest list
        timestamps_shortest_list = [x[0] for x in shortest_list]
        print(f"\nTimestamps of the shortest list: {timestamps_shortest_list}")

        # get the difference between the timestamps of the common part and the shortest list
        timestamps_differences = [
            timestamps_common_part[i] - timestamps_shortest_list[i]
            for i in range(len(shortest_list))
        ]
        print(
            f"\nDifference between the timestamps of the common part and the shortest list: {timestamps_differences}"
        )
        # print the mean and the standard deviation of the timestamps difference
        timestamps_diff_mean = np.mean(timestamps_differences)
        timestamps_diff_std = np.std(timestamps_differences)
        print(f"\nMean of the timestamps difference: {timestamps_diff_mean:.3f}")
        print(
            f"Standard deviation of the timestamps difference: {timestamps_diff_std:.3f}"
        )


if doRunTests:

    def test_Delay_Results_for_xdf(visit_path):
        xdf_csv_files = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_csv_files:
            delay_results = Delay_Results_for_xdf(xdf_csv)
            delay_results.summary()

    test_Delay_Results_for_xdf(xdf_path)

## Compute the kinect-to-csv delay

We want to compute the delay between the kinect stream and the kinect csv file. 
This case is the simplest, as we have only one `*_k_m*.csv` file corresponding to only one kinect stream.  
Both contain the same markers sequences, but with different timestamps, and maybe with different markers in the beg and the end of the sequences (due to different start-stop for the xdf and csv records).

The steps are:
- Define the shortest and the longest marker list from the two marker lists.
- Find the (first) occurrence of the shortest marker list in the longest marker list. 
- Compute the delay as the difference between the timestamps of the two occurrences.


In the kinect mocap csv files, the data is :

```
Software : LSL_Kinect,Version : 1.0.5.2,Stream nominal rate : 15,Sequence Name : Reaching Task

TimeSpan            ,SpineBase_X         ,SpineBase_Y,          SpineBase_Z        ,SpineBase_Conf, ...
1616421533460.0652  ,0.22550877928733826 ,-0.13375791907310486, 2.5088465213775635 ,1             , ...
1616421533497.9622  ,0.22533915936946869 ,-0.13381971418857575 ,2.5087275505065918 ,1             , ...
...
```

TimeSpan is the timestamp in milliseconds.

In the kinect mocap stream in the xdf file, the data is :
- `xdf_kinect_mocap["time_stamps"]`: timestamps in the xdf file
  * ** NOTE ** : the timestamps in version <= 1.0.5.2 are INCORRECT, they are relative to the computer's startup time, whereas timestamps should be defined by LSL to enable synchronization with other streams.

- `xdf_kinect_mocap["time_series"]`: an exact copy of the kinect mocap csv file (i.e., the first column is the TimeSpan column)
  * ** NOTE ** : the timestamps in the TimeSpan column are the timestamps in the kinect mocap csv file.

As a consequence, we can use the TimeSpan column to get the timestamps in the kinect mocap csv file, so to compute the delay between the kinect stream and the kinect csv file.

```python
...
xdf_kinect_timestamps = xdf_kinect_mocap["time_stamps"]
csv_kinect_timestamps = [float(row[0]) for row in kinect_csv_data]
kinect_to_csv_delay = xdf_kinect_timestamps - csv_kinect_timestamps

...
```



In [ ]:
class Delay_Results_by_xdf:

    def __init_all_var__(self):
        """Initialize all variables to None or empty"""
        self.xdf_csv_files = {}
        self.pattern = ""
        self.delays = None
        self.xdf_kinect_mocap = None
        self.xdf_kinect_markers = None
        self.xdf_mouse_markers = None
        self.markers_csv_list = None
        self.markers_xdf_list = None
        self.kinect_to_csv_delay = None

    def __init__(self, xdf_csv_files, pattern):
        self.__init_all_var__()
        self.xdf_csv_files = xdf_csv_files
        self.pattern = pattern
        self.delays = Delay_Results_for_xdf(xdf_csv_files)
        if not self.delays.mandatory_files_error:
            self.__set_xdf__()
            self.__set_kinect_to_csv_delay__()
            self.__set_lists_to_compare__()
            self.__compare_lists__()

    def __set_xdf__(self):
        """Set the xdf data"""
        xdf_fpath = os.path.join(xdf_path, self.xdf_csv_files["xdf"])
        k_mo, k_mk, m_mk = load_xdf_kinect_mouse_streams(xdf_fpath)
        self.xdf_kinect_mocap = k_mo
        self.xdf_kinect_markers = k_mk
        self.xdf_mouse_markers = m_mk

    def __set_kinect_to_csv_delay__(self):
        """Set the delay between the kinect and the csv files"""
        if not self.xdf_kinect_mocap:
            return
        xdf_kinect_timestamps = self.xdf_kinect_mocap["time_stamps"]
        kinect_csv_data = self.xdf_kinect_mocap["time_series"]
        csv_kinect_timestamps = kinect_csv_data[:, 0]
        csv_kinect_timestamps = csv_kinect_timestamps.astype(float) / 1000
        self.kinect_to_csv_delay = xdf_kinect_timestamps - csv_kinect_timestamps

        if do_debug:
            import matplotlib.pyplot as plt

            plt.title(
                f"Kinect to CSV delay, mean = {np.mean(self.kinect_to_csv_delay):.3f}"
            )
            plt.scatter(xdf_kinect_timestamps, self.kinect_to_csv_delay)
            plt.xlabel("Kinect timestamp")
            plt.ylabel("Kinect to CSV delay")
            plt.show()

    def __set_mouse_to_csv_delay__(self):
        """Set the delay between the mouse and the csv files"""
        self.markers_xdf_list = marker_list(self.xdf_mouse_markers)
        self.markers_csv_list = get_all_csv_marker_of_type_list(
            self.xdf_csv_files["csv"],
            "_l_m",
        )
        self.__compare_lists__()

    def __set_lists_to_compare__(self):
        """
        Set the lists to compare
        """

        if self.pattern == "_l_m":
            self.markers_xdf_list = marker_list(self.xdf_mouse_markers)
        elif self.pattern == "_k_m":
            self.markers_xdf_list = marker_list(self.xdf_kinect_markers)
        elif self.pattern == "_k.":
            self.markers_xdf_list = self.create_kinect_mocap_xdf_list()
        else:
            raise ValueError("The pattern must be '_l_m' or '_k_m' or '_k.'")

        if self.pattern == "_k.":
            self.markers_csv_list = self.create_kinect_mocap_csv_list()
        else:
            self.markers_csv_list = get_all_csv_marker_of_type_list(
                self.xdf_csv_files["csv"], self.pattern
            )

    def __compare_lists__(self):
        """Compare the lists"""

        shortest_list, longest_list = define_shortest_longest_lists(
            self.markers_xdf_list, self.markers_csv_list
        )

        i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
            shortest_list, longest_list
        )

        timestamps_differences = get_timestamps_differences(
            longest_list, shortest_list, i_beg, i_end
        )

        if self.delays:
            self.delays.results["error_msg"] = error_msg
            self.delays.results["timestamps_differences"] = timestamps_differences
            self.delays.results["iBeg"] = i_beg
            self.delays.results["iEnd"] = i_end
            self.delays.results["longest_list"] = longest_list
            self.delays.results["shortest_list"] = shortest_list
            self.delays.results["markers_csv_list"] = self.markers_csv_list
            self.delays.results["markers_xdf_list"] = self.markers_xdf_list
            self.delays.results["from_marker"] = self.pattern

    def summary(self):
        """Print the marker delays results summary"""
        if self.delays:
            self.delays.summary()

    def create_kinect_mocap_xdf_list(self):
        """Create a list of [timestamp, markers] from the kinect mocap stream,
        where the marker is the first 5 columns values joined as a string
        """
        if not self.xdf_kinect_mocap:
            return []

        kinect_mocap_timeseries = self.xdf_kinect_mocap["time_series"]
        kinect_mocap_timestamps = self.xdf_kinect_mocap["time_stamps"]

        kinect_mocap_xdf_list = []
        nb_li = len(kinect_mocap_timeseries)
        for i in range(nb_li):
            markers = [kinect_mocap_timeseries[i][j] for j in range(5)]
            markers = [str(marker) for marker in markers]
            markers = ",".join(markers)
            timestamp = kinect_mocap_timestamps[i]

            to_add = [timestamp, markers]
            kinect_mocap_xdf_list.append(to_add)

        print(f"kinect_mocap_timeseries.shape: {kinect_mocap_timeseries.shape}")
        # print the kinect_mocap_xdf_list length and the first element
        print(f"len(kinect_mocap_xdf_list): {len(kinect_mocap_xdf_list)}")
        print(f"kinect_mocap_xdf_list[0]: {kinect_mocap_xdf_list[0]}")

        k_time = kinect_mocap_timestamps

        c_time = kinect_mocap_timeseries[:, 0]
        c_time = c_time.astype(float) / 1000  # convert to seconds

        print(f"c_time[0]: {c_time[0]}")
        print(f"k_time[0]: {k_time[0]}")

        d_time = c_time - k_time

        # make a boxplot of the differences
        import matplotlib.pyplot as plt

        plt.boxplot(d_time - d_time[0])
        plt.show()

        return kinect_mocap_xdf_list

    def create_kinect_mocap_csv_list(self):
        """Create a list of [timestamp, markers] from the kinect mocap csv files,
        where the marker is the first 5 columns values joined as a string
        """

        kinect_mocap_csv_list = []
        for csv in self.xdf_csv_files["csv"]:
            if "_k." in csv:
                # load the csv file
                lines = np.loadtxt(
                    fname=os.path.join(xdf_path, csv),
                    skiprows=3,
                    delimiter=",",
                    quotechar='"',
                    dtype=str,
                )
                # keep only the first 5 columns
                lines = np.delete(lines, np.s_[5:], 1)

                k_time = lines[:, 0]
                k_time = k_time.astype(float)

                # create a list of [timestamp, markers]
                for line in lines:
                    # timestamp is the first column (in milliseconds) => convert to seconds
                    timestamp = float(line[0]) / 1000
                    # markers are the other columns => joined as a string
                    markers = ",".join(line[1:])
                    kinect_mocap_csv_list.append([timestamp, markers])

                print(f"lines.shape: {lines.shape}")
                # print the kinect_mocap_xdf_list length and the first element
                print(f"len(kinect_mocap_csv_list): {len(kinect_mocap_csv_list)}")
                print(f"kinect_mocap_csv_list[0]: {kinect_mocap_csv_list[0]}")

                timestamp_difference = (
                    kinect_mocap_csv_list[0][0]
                    - self.xdf_kinect_mocap["time_stamps"][0]
                )
                print(f"timestamp_difference: {timestamp_difference}")

                a = 1617972055.108
                print(f"a: {a}")
                print(f"b: {timestamp_difference}")

                print(f"a-timestamp_difference: {a-timestamp_difference}")

                kinect_to_csv_timestamp_delta = 1617972055.108 - timestamp_difference

                a = 0

        return kinect_mocap_csv_list


if doRunTests:

    def test_Delay_Results_by_xdf(visit_path):
        xdf_csv_files = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_csv_files:
            delay_results = Delay_Results_by_xdf(xdf_csv, "_k.")
            delay_results.summary()

    test_Delay_Results_by_xdf(xdf_path)

In [ ]:
# TODO: add this into the class Delay_Results_for_xdf?
# Reason:  we use similar code for the kinect markers and the mouse markers
#         and we shall use the same code for the kinect mocap markers
# How: the pattern of the marker name ("_k_m", "_l_m") can distinguish between the different types of markers


def get_xdf_to_csv_kinect_marker_delay_class(xdf_csv):
    """Get the delays between the kinect markers from the xdf file and the csv files.
    return a Delay_Results_for_xdf object"""

    x2c_delays = Delay_Results_for_xdf(xdf_csv)

    if x2c_delays.mandatory_files_error:
        return x2c_delays

    kinect_mocap, kinect_markers, mouse_markers = load_xdf_kinect_mouse_streams(
        os.path.join(xdf_path, x2c_delays.xdf_rel_fname)
    )

    kinect_markers_csv_all_list = get_all_csv_marker_of_type_list(
        x2c_delays.csv_rel_fnames, "_k_m"
    )

    kinect_markers_xdf_list = marker_list(kinect_markers)

    shortest_list, longest_list = define_shortest_longest_lists(
        kinect_markers_xdf_list, kinect_markers_csv_all_list
    )

    i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
        shortest_list, longest_list
    )

    if error_msg:
        x2c_delays.results["error_msg"] = error_msg
        return x2c_delays

    timestamps_differences = get_timestamps_differences(
        longest_list, shortest_list, i_beg, i_end
    )

    x2c_delays.results["timestamps_differences"] = timestamps_differences
    x2c_delays.results["iBeg"] = i_beg
    x2c_delays.results["iEnd"] = i_end
    x2c_delays.results["longest_list"] = longest_list
    x2c_delays.results["shortest_list"] = shortest_list
    x2c_delays.results["markers_csv_list"] = kinect_markers_csv_all_list
    x2c_delays.results["markers_xdf_list"] = kinect_markers_xdf_list
    x2c_delays.results["from_marker"] = "kinect"

    return x2c_delays


if doRunTests:

    def test_get_xdf_to_csv_kinect_marker_delay_class(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_and_csv_files_in_visit:
            x2c_delays = get_xdf_to_csv_kinect_marker_delay_class(xdf_csv)
            x2c_delays.summary()

    test_get_xdf_to_csv_kinect_marker_delay_class(xdf_path)

## Compute the mouse-to-csv delay

We want to compute the delay between the mouse stream and the mouse csv files.  
This case is more complex, as we have multiple `*_l_m*.csv` files corresponding to only one the mouse stream,and maybe with different markers in the beg and the end of the sequences (due to different start-stop for the xdf and csv records).

The steps are:
- Define the shortest and the longest marker list from the two marker lists.
- Find the (first) occurrence of the shortest marker list in the longest marker list. 
- Compute the delay as the difference between the timestamps of the two occurrences.

In [ ]:
def get_xdf_to_csv_mouse_marker_delay_class(xdf_csv):
    """Get the delays between the mouse markers from the xdf file and the csv files.
    return a Delay_Results_for_xdf object"""

    x2c_delays = Delay_Results_for_xdf(xdf_csv)

    if x2c_delays.mandatory_files_error:
        return x2c_delays

    kinect_mocap, kinect_markers, mouse_markers = load_xdf_kinect_mouse_streams(
        os.path.join(xdf_path, x2c_delays.xdf_rel_fname)
    )

    mouse_markers_csv_all_list = get_all_csv_marker_of_type_list(
        x2c_delays.csv_rel_fnames, "_l_m"
    )
    mouse_markers_xdf_list = marker_list(mouse_markers)

    shortest_list, longest_list = define_shortest_longest_lists(
        mouse_markers_xdf_list, mouse_markers_csv_all_list
    )

    i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
        shortest_list, longest_list
    )

    if error_msg:
        x2c_delays.results["error_msg"] = error_msg
        return x2c_delays

    timestamps_differences = get_timestamps_differences(
        longest_list, shortest_list, i_beg, i_end
    )

    x2c_delays.results["timestamps_differences"] = timestamps_differences
    x2c_delays.results["iBeg"] = i_beg
    x2c_delays.results["iEnd"] = i_end
    x2c_delays.results["longest_list"] = longest_list
    x2c_delays.results["shortest_list"] = shortest_list
    x2c_delays.results["markers_csv_list"] = mouse_markers_csv_all_list
    x2c_delays.results["markers_xdf_list"] = mouse_markers_xdf_list
    x2c_delays.results["from_marker"] = "mouse"

    return x2c_delays


if doRunTests:

    def test_get_xdf_to_csv_mouse_marker_delay_class(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_and_csv_files_in_visit:
            x2c_delays = get_xdf_to_csv_mouse_marker_delay_class(xdf_csv)
            x2c_delays.summary()

    test_get_xdf_to_csv_mouse_marker_delay_class(xdf_path)

In [ ]:
def get_xdf_to_csv_mouse_marker_delay_dict(xdf_csv):
    """Get the dictionary of the delays between the xdf and the mouse csv markers."""

    xdf_rel_fname = xdf_csv["xdf"]
    csv_rel_fnames = xdf_csv["csv"]
    mandatory_files_error = xdf_csv["mandatory_files_error"]

    if mandatory_files_error:
        return {
            "error_msg": mandatory_files_error,
            "timestamps_differences": [],
        }

    kinect_mocap, kinect_markers, mouse_markers = load_xdf_kinect_mouse_streams(
        os.path.join(xdf_path, xdf_rel_fname)
    )

    mouse_markers_csv_list = get_all_csv_marker_of_type_list(csv_rel_fnames, "_l_m")
    mouse_markers_xdf_list = marker_list(mouse_markers)

    shortest_list, longest_list = define_shortest_longest_lists(
        mouse_markers_xdf_list, mouse_markers_csv_list
    )

    i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
        shortest_list, longest_list
    )

    if error_msg:
        return {
            "error_msg": error_msg,
            "timestamps_differences": [],
        }

    timestamps_differences = get_timestamps_differences(
        longest_list, shortest_list, i_beg, i_end
    )

    return {
        "error_msg": error_msg,
        "timestamps_differences": timestamps_differences,
        "iBeg": i_beg,
        "iEnd": i_end,
        "longest_list": longest_list,
        "shortest_list": shortest_list,
        "markers_csv_list": mouse_markers_csv_list,
        "markers_xdf_list": mouse_markers_xdf_list,
        "from_marker": "mouse",
    }


def print_marker_list_delays(xdf_to_csv_marker_delay_dict):
    """Print the marker list delays in the xdf_to_csv_marker_delay_dict"""

    error_msg = xdf_to_csv_marker_delay_dict["error_msg"]
    if error_msg:
        print(f"  >>>>>> ERROR: {error_msg}")
    else:
        # get the values from the dictionary
        timestamps_differences = xdf_to_csv_marker_delay_dict["timestamps_differences"]
        i_beg = xdf_to_csv_marker_delay_dict["iBeg"]
        i_end = xdf_to_csv_marker_delay_dict["iEnd"]
        longest_list = xdf_to_csv_marker_delay_dict["longest_list"]
        shortest_list = xdf_to_csv_marker_delay_dict["shortest_list"]
        markers_csv_list = xdf_to_csv_marker_delay_dict["markers_csv_list"]
        markers_xdf_list = xdf_to_csv_marker_delay_dict["markers_xdf_list"]
        from_marker = xdf_to_csv_marker_delay_dict["from_marker"]

        # start printing
        print("")
        print_marker_list(
            markers_csv_list, f"Sorted {from_marker} markers from csv files"
        )
        print_marker_list(markers_xdf_list, f"{from_marker} markers from xdf")

        print("")
        print(f"First occurrence of the shortest list in the longest list: {i_beg}")
        print(f"Last occurrence of the shortest list in the longest list: {i_end}")

        # print the common part of the longest list
        common_part = longest_list[i_beg : i_end + 1]
        print_marker_list(common_part, "Common part of the longest list")

        # print the timestamps of the common part
        timestamps_common_part = [x[0] for x in common_part]
        print(f"\nTimestamps of the common part: {timestamps_common_part}")

        # print the timestamps of the shortest list
        timestamps_shortest_list = [x[0] for x in shortest_list]
        print(f"\nTimestamps of the shortest list: {timestamps_shortest_list}")

        # get the difference between the timestamps of the common part and the shortest list
        timestamps_differences = [
            timestamps_common_part[i] - timestamps_shortest_list[i]
            for i in range(len(shortest_list))
        ]
        print(
            f"\nDifference between the timestamps of the common part and the shortest list: {timestamps_differences}"
        )
        # print the mean and the standard deviation of the timestamps difference
        timestamps_diff_mean = np.mean(timestamps_differences)
        timestamps_diff_std = np.std(timestamps_differences)
        print(f"\nMean of the timestamps difference: {timestamps_diff_mean:.3f}")
        print(
            f"Standard deviation of the timestamps difference: {timestamps_diff_std:.3f}"
        )


if doRunTests:

    def test_get_xdf_to_csv_mouse_marker_delay_dict(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_and_csv_files_in_visit:
            xdf_to_csv_mouse = get_xdf_to_csv_mouse_marker_delay_dict(xdf_csv)

            print(
                "\n*****************************************************************************"
                + "\n*****************************************************************************"
            )

            xdf_rel_fname = xdf_csv["xdf"]
            print(f"\nXDF: {xdf_rel_fname}")
            print_marker_list_delays(xdf_to_csv_mouse)

    test_get_xdf_to_csv_mouse_marker_delay_dict(xdf_path)

## Compute the kinect-to-mouse delay

In [ ]:
def get_kinect_to_mouse_markers_delay_for_one_xdf_class(xdf_csv):
    """Get the delay between the kinect and the mouse markers from one xdf file and the corresponding csv files"""

    mouse_to_csv_delay = get_xdf_to_csv_mouse_marker_delay_class(xdf_csv)

    kinect_to_cvs_delay = get_xdf_to_csv_kinect_marker_delay_class(xdf_csv)

    if (
        kinect_to_cvs_delay.results["error_msg"]
        or mouse_to_csv_delay.results["error_msg"]
    ):
        return {
            "kinect_to_cvs_delay_list": [],
            "mouse_to_csv_delay_list": [],
            "kinect_to_mouse_delay_list": [],
            "kinect_to_mouse_delay_mean": np.nan,
            "kinect_to_mouse_delay_std": np.nan,
            "error_msg": "An error occurred when computing the delays."
            + kinect_to_cvs_delay.results["error_msg"]
            + mouse_to_csv_delay.results["error_msg"],
        }

    kinect_to_cvs_delay_mean = np.mean(
        kinect_to_cvs_delay.results["timestamps_differences"]
    )
    kinect_to_cvs_delay_std = np.std(
        kinect_to_cvs_delay.results["timestamps_differences"]
    )

    mouse_to_csv_delay_mean = np.mean(
        mouse_to_csv_delay.results["timestamps_differences"]
    )
    mouse_to_csv_delay_std = np.std(
        mouse_to_csv_delay.results["timestamps_differences"]
    )

    # NOTE: the kinect to mouse delay is the difference of the means
    # The mean & std is the simplest approach, if the distribution is normal
    # which should be true, as the error is due to random delays in the system
    kinect_to_mouse_delay_mean = kinect_to_cvs_delay_mean - mouse_to_csv_delay_mean

    kinect_to_mouse_delay_median = np.median(
        kinect_to_cvs_delay.results["timestamps_differences"]
    ) - np.median(mouse_to_csv_delay.results["timestamps_differences"])

    # NOTE: the variance of the difference is the sum of the variances **minus the covariance**
    # (e.g. https://en.wikipedia.org/wiki/Propagation_of_uncertainty)
    # If we assume that the two distributions are independent, the covariance is zero
    # hence computing the variance of the difference as the sum of the variances is correct.
    # If we assume that the two distributions are not independent, we should subtract the covariance,
    # but we do not have it. We still can compute the variance of the difference as the sum of the variances,
    # and this will be an **upper bound of the variance of the difference**.
    kinect_to_mouse_delay_std = np.sqrt(
        kinect_to_cvs_delay_std**2 + mouse_to_csv_delay_std**2
    )

    kinect_to_cvs_delay_list = kinect_to_cvs_delay.results["timestamps_differences"]
    mouse_to_csv_delay_list = mouse_to_csv_delay.results["timestamps_differences"]

    return {
        "kinect_to_cvs_delay_list": kinect_to_cvs_delay_list,
        "mouse_to_csv_delay_list": mouse_to_csv_delay_list,
        "kinect_to_mouse_delay_mean": kinect_to_mouse_delay_mean,
        "kinect_to_mouse_delay_median": kinect_to_mouse_delay_median,
        "kinect_to_mouse_delay_std": kinect_to_mouse_delay_std,
        "error_msg": "",
    }


def print_kinect_to_mouse_delay_dict(ktm_delays):
    """Print the kinect to mouse delay dictionary"""

    kinect_to_cvs_delay_list = ktm_delays["kinect_to_cvs_delay_list"]
    mouse_to_csv_delay_list = ktm_delays["mouse_to_csv_delay_list"]
    kinect_to_mouse_delay_mean = ktm_delays["kinect_to_mouse_delay_mean"]
    kinect_to_mouse_delay_median = ktm_delays["kinect_to_mouse_delay_median"]
    kinect_to_mouse_delay_std = ktm_delays["kinect_to_mouse_delay_std"]
    error_msg = ktm_delays["error_msg"]

    if error_msg:
        print(f"\nError: {error_msg}")
        return

    # we need this local import to avoid conflict with global import:
    # from datetime import datetime (for the function check_csv_date)
    from datetime import timedelta

    kinect_to_mouse_delay_mean_HMS = timedelta(seconds=abs(kinect_to_mouse_delay_mean))

    print(f"\nKinect to mouse delays: ")
    print(f"  - kinect to cvs: {kinect_to_cvs_delay_list}")
    print(f"  - mouse to cvs: {mouse_to_csv_delay_list}")
    print(
        f"  - kinect to mouse delay mean = {kinect_to_mouse_delay_mean:.3f} s"
        + f" ({kinect_to_mouse_delay_mean_HMS})"
    )
    print(f"  - kinect to mouse delay std < {kinect_to_mouse_delay_std:.6f} s")

    print(f"  - kinect to mouse delay median = {kinect_to_mouse_delay_median:.3f} s")

    ktm_mean_median_diff = kinect_to_mouse_delay_mean - kinect_to_mouse_delay_median

    print(f"  - kinect to mouse delay mean - median = {ktm_mean_median_diff:.3f} s")

    return


def analyse_ktm_delay_for_one_xdf(xdf_csv):
    """Analyse the kinect to mouse delay for one xdf file"""

    ktm_delay_for_one_xdf = get_kinect_to_mouse_markers_delay_for_one_xdf_class(xdf_csv)

    xdf_rel_fname = xdf_csv["xdf"]
    print("")
    print(
        "###############################################################################"
    )
    print(f"XDF: {xdf_rel_fname}")
    if ktm_delay_for_one_xdf["error_msg"]:
        print(f"\nError: {ktm_delay_for_one_xdf['error_msg']}")
        return

    k_delays = ktm_delay_for_one_xdf["kinect_to_cvs_delay_list"]
    # kk = get_xdf_to_csv_kinect_marker_delay_dict(xdf_rel_fname, csv_rel_fnames)

    # k_delays_diff = k_delays["timestamps_differences"]
    print(f"\nKinect to csv delays: {np.mean(k_delays)}")

    # m_delays = get_xdf_to_csv_mouse_marker_delay_dict(xdf_rel_fname, csv_rel_fnames)
    # m_delays_diff = m_delays["timestamps_differences"]
    m_delays = ktm_delay_for_one_xdf["mouse_to_csv_delay_list"]
    print(f"\nMouse to csv delays: {np.mean(m_delays)}")

    # ktm_delay = np.mean(k_delays_diff) - np.mean(m_delays_diff)
    ktm_delay = np.mean(k_delays) - np.mean(m_delays)
    print(f"\nKinect to mouse delay: {ktm_delay}")

    np.set_printoptions(precision=3)

    # kinect_markers_timestamps = kinect_markers["time_stamps"]
    # print(f"\nK ts     : {kinect_markers_timestamps}")

    # new_kinect_markers_timestamps = kinect_markers["time_stamps"] - ktm_delay
    # print(f"\nK ts new : {new_kinect_markers_timestamps}")

    # mouse_markers_timestamps = mouse_markers["time_stamps"]
    # print(f"\nM ts     : {mouse_markers_timestamps[:5]}...")

    # analysis of the delay distribution
    ################################################################################################
    ktm_delays = get_kinect_to_mouse_markers_delay_for_one_xdf_class(xdf_csv)
    ################################################################################################
    kinect_to_cvs_delay_list = ktm_delays["kinect_to_cvs_delay_list"]
    mouse_to_csv_delay_list = ktm_delays["mouse_to_csv_delay_list"]

    # kinect_to_mouse_delay_mean = ktm_delays["kinect_to_mouse_delay_mean"]
    # kinect_to_mouse_delay_std = ktm_delays["kinect_to_mouse_delay_std"]

    print_kinect_to_mouse_delay_dict(ktm_delays)

    # make two boxplot of the distribution of the delays (kinect to cvs and mouse to cvs) in 2 figures
    # and add the values of each delay in the boxplot
    import matplotlib.pyplot as plt

    # make a boxplot of the centered distribution of the kinect and mouse delays on a single figure
    kinect_to_cvs_delay_list_centered = kinect_to_cvs_delay_list - np.mean(
        kinect_to_cvs_delay_list
    )
    mouse_to_csv_delay_list_centered = mouse_to_csv_delay_list - np.mean(
        mouse_to_csv_delay_list
    )

    fig, axs = plt.subplots(1, 1, figsize=(5, 5))
    axs.boxplot(
        [
            kinect_to_cvs_delay_list_centered,
            mouse_to_csv_delay_list_centered,
        ]
    )
    # set the ylimits
    yl = axs.get_ylim()
    min_y = 0.001
    if np.abs(yl).max() < min_y:
        axs.set_ylim(-min_y, min_y)
    # make the limits symmetrical
    axs.set_ylim(-np.abs(yl).max(), np.abs(yl).max())

    # add a dot for each value
    axs.scatter(
        [1] * len(kinect_to_cvs_delay_list_centered),
        kinect_to_cvs_delay_list_centered,
        color="blue",
        alpha=0.6,
    )
    axs.scatter(
        [2] * len(mouse_to_csv_delay_list_centered),
        mouse_to_csv_delay_list_centered,
        color="blue",
        alpha=0.6,
    )
    # add a horizontal line for the mean and label it with the mean value
    axs.axhline(
        np.mean(kinect_to_cvs_delay_list_centered), color="grey", linestyle="--"
    )
    axs.text(
        1.5,
        np.mean(kinect_to_cvs_delay_list_centered),
        f"mean",
        fontsize=12,
        color="grey",
        ha="center",
        va="center",
        backgroundcolor="w",
    )

    axs.set_title(f"Kinect to cvs and mouse to cvs delays; fname {xdf_rel_fname}")
    axs.set_xticklabels(["Kinect to cvs", "Mouse to cvs"])
    axs.set_ylabel("Delay - mean (s)")
    plt.show()


if doRunTests:

    def test_analyse_ktm_delay_for_one_xdf(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_and_csv_files_in_visit:
            analyse_ktm_delay_for_one_xdf(xdf_csv)

    test_analyse_ktm_delay_for_one_xdf(xdf_path)

## Compare the timestamps of the kinect mocap from the csv and from the xdf file

Sometimes, the kinect markers are not correctly recorded in the xdf file (due to a bug in LSL_Kinect)... 
Consequently, we cannot rely on the kinect markers only. Too bad...

Solution: *if we have the kinect mocap csv file*
- We can use the timestamps of the kinect mocap from the csv file to get another view the kinect-to-csv delay.   
- We can then verify that the kinect-to-csv delay from the markers is consistent with the kinect-to-csv delay from the mocap.



In [ ]:
def create_kinect_mocap_xdf_list(kinect_mocap):
    """Create a list of [timestamp, markers] from the kinect mocap stream,
    where the marker is the first 5 columns values joined as a string
    """

    kinect_mocap_timeseries = kinect_mocap["time_series"]
    kinect_mocap_timestamps = kinect_mocap["time_stamps"]

    kinect_mocap_xdf_list = []
    nb_li = len(kinect_mocap_timeseries)
    for i in range(nb_li):
        markers = [kinect_mocap_timeseries[i][j] for j in range(5)]
        markers = [str(marker) for marker in markers]
        markers = ",".join(markers)
        timestamp = kinect_mocap_timestamps[i]
        kinect_mocap_xdf_list.append([timestamp, markers])

    return kinect_mocap_xdf_list


if doRunTests:

    def test_create_kinect_mocap_xdf_list(visit_path):
        xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
        for xdf_csv in xdf_and_csv_files_in_visit:
            csv_rel_fnames = xdf_csv["csv"]

            print(
                "\n*****************************************************************************"
            )

            kinect_mocap, kinect_markers, mouse_markers = load_xdf_kinect_mouse_streams(
                os.path.join(visit_path, xdf_csv["xdf"])
            )

            kinect_mocap_csv_list_xxx = get_all_csv_marker_of_type_list(
                csv_rel_fnames, "_k.csv"
            )

            kinect_mocap_xdf_list_xxx = marker_list(kinect_mocap)

            # verify that the lists are the same length
            # it is not the case if some csv files are missing...
            if len(kinect_mocap_csv_list_xxx) != len(kinect_mocap_xdf_list_xxx):
                print(
                    f"The lists are not of the same length: {len(kinect_mocap_csv_list_xxx)} != {len(kinect_mocap_xdf_list_xxx)}"
                )

            # decode a kinect mocap stream from the xdf file
            kinect_mocap_timeseries = kinect_mocap["time_series"]
            kinect_mocap_timestamps = kinect_mocap["time_stamps"]
            # get the names of the first 5 columns from the kinect_mocap['info']['desc'][0]['channels'][0]
            kinect_mocap_channels = kinect_mocap["info"]["desc"][0]["channels"][0]
            kinect_mocap_channel_names_list = kinect_mocap_channels["channel"]
            # print the first 5 labels
            print(f"\nKinect mocap stream labels: ")
            for i in range(5):
                label = kinect_mocap_channel_names_list[i]["label"][0]
                print(f"{i}: {label}")

            ################################################################################################
            kinect_mocap_xdf_list = create_kinect_mocap_xdf_list(kinect_mocap)

            # verify that the lists are the same length
            if len(kinect_mocap_csv_list_xxx) != len(kinect_mocap_xdf_list_xxx):
                print("ERROR : The lists are not of the same length")

            # print the first 5 markers
            print(f"\nKinect mocap markers: {len(kinect_mocap_xdf_list)} markers")
            for i in range(5):
                print(f"{i}: {kinect_mocap_xdf_list[i]}")

    test_create_kinect_mocap_xdf_list(xdf_path)

In [ ]:
def create_kinect_mocap_csv_list(csv_rel_fnames):
    """Create a list of [timestamp, markers] from the kinect mocap csv file,
    where the marker is the first 5 columns values joined as a string
    """

    kinect_mocap_csv_fname = [csv for csv in csv_rel_fnames if "_k.csv" in csv]
    kinect_mocap_csv_fname = kinect_mocap_csv_fname[0]
    kinect_mocap_csv_fname = os.path.join(xdf_path, kinect_mocap_csv_fname)

    with open(kinect_mocap_csv_fname, "r") as fname:
        txt = fname.readlines()

    lines = [line.split(",") for line in txt]

    # remove the first 3 lines (header)
    lines = lines[3:]

    # keep only the first 5 columns
    kinect_mocap_csv_list = []
    for line in lines:
        markers = [line[j] for j in range(5)]
        markers = [float(marker) for marker in markers]  # float (as in xdf)
        markers = [str(marker) for marker in markers]
        markers = ",".join(markers)
        timestamp = float(line[0]) / 1000
        kinect_mocap_csv_list.append([timestamp, markers])

    return kinect_mocap_csv_list


def analyse_kinect_mocap_csv_list(csv_rel_fnames):
    ###########################################################################################
    kinect_mocap_csv_list = create_kinect_mocap_csv_list(csv_rel_fnames)

    # print the first 5 markers
    print(f"\nKinect mocap markers from csv: ")
    for i in range(5):
        print(f"{i}: {kinect_mocap_csv_list[i]}")


if doRunTests:
    xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(xdf_path)
    for xdf_csv in xdf_and_csv_files_in_visit:
        xdf_rel_fname = xdf_csv["xdf"]
        csv_rel_fnames = xdf_csv["csv"]
        mandatory_files_error = xdf_csv["mandatory_files_error"]
        expected_files_warning = xdf_csv["expected_files_warning"]

        print(
            "\n*****************************************************************************"
            + "\n*****************************************************************************"
        )
        print(f"\nXDF: {xdf_rel_fname}")

        if mandatory_files_error:
            print(f"  >>>>>> ERROR: {mandatory_files_error}")
        else:
            # do the analysis
            analyse_kinect_mocap_csv_list(csv_rel_fnames)

### Verify that the kinect-to-csv delay from the markers is consistent with the kinect-to-csv delay from the mocap

The logic is to compare the kinect-to-csv delay from the markers with the kinect-to-csv delay from the mocap.


In [ ]:
# get the kinect-to-csv delay from the markers

In [ ]:
def get_kinect_mocap_csv_fullfname(xdf_csv_files):
    """Get the full filename of the kinect mocap csv file"""

    kinect_mocap_csv_fname = [csv for csv in xdf_csv_files if "_k.csv" in csv]
    if len(kinect_mocap_csv_fname) == 0:
        raise ValueError(
            "There is no kinect mocap csv file (with '_k.csv' in the name)"
        )
    if len(kinect_mocap_csv_fname) != 1:
        raise ValueError(
            "There should be only one kinect mocap csv file (with '_k.csv' in the name)"
        )

    kinect_mocap_csv_fname = kinect_mocap_csv_fname[0]
    kinect_mocap_csv_fname = os.path.join(xdf_path, kinect_mocap_csv_fname)

    return kinect_mocap_csv_fname


def verify_kinect_to_csv_mocap_delay(xdf_rel_fname, csv_rel_fnames):
    """Verify that the kinect-to-csv delay from the markers is consistent with the kinect-to-csv delay from the mocap"""

    kinect_mocap_csv_full_fname = get_kinect_mocap_csv_fullfname(csv_rel_fnames)

    kinect_mocap_xdf_list = create_kinect_mocap_xdf_list(kinect_mocap)
    kinect_mocap_csv_list = create_kinect_mocap_csv_list(csv_rel_fnames)

    # verify that the lists are the same length
    # NOTE: this should not happen, but it is better to check...

    kinect_mocap_timestamps_diff_mean = np.nan
    kinect_mocap_timestamps_diff_std = np.nan
    warning_msg = ""
    error_msg = ""

    kinect_mocap_xdf_list_ok = kinect_mocap_xdf_list
    kinect_mocap_csv_list_ok = kinect_mocap_csv_list
    if len(kinect_mocap_csv_list) != len(kinect_mocap_xdf_list):
        warning_msg += "\n" + (
            f"Different length: mocap-cvs={len(kinect_mocap_csv_list)} vs mocap-xdf={len(kinect_mocap_xdf_list)}"
        )
        # print the filename of the csv file and the xdf file
        warning_msg += "\n" + (f"csv: {kinect_mocap_csv_full_fname}")
        warning_msg += "\n" + (f"xdf: {xdf_rel_fname}")

        shortest_list, longest_list = define_shortest_longest_lists(
            kinect_mocap_xdf_list, kinect_mocap_csv_list
        )
        i_beg, i_end, error_in_find = find_first_occurrence_of_short_list_in_long_list(
            shortest_list, longest_list
        )

        warning_msg += "\n" + (
            f"First occurrence of the shortest list in the longest list: {i_beg}"
        )
        warning_msg += "\n" + (
            f"Last occurrence of the shortest list in the longest list: {i_end}"
        )

        if shortest_list == kinect_mocap_xdf_list:
            kinect_mocap_csv_list_ok = kinect_mocap_csv_list[i_beg : i_end + 1]
            kinect_mocap_xdf_list_ok = kinect_mocap_xdf_list
        else:
            kinect_mocap_xdf_list_ok = kinect_mocap_xdf_list[i_beg : i_end + 1]
            kinect_mocap_csv_list_ok = kinect_mocap_csv_list

        if len(kinect_mocap_xdf_list_ok) != len(kinect_mocap_csv_list_ok):
            # this should not happen... but... just in case...
            error_msg += "\n" + (
                f"Could not find a partial match between mocap-cvs and mocap-xdf lists"
            )
            error_msg += "\n" + (
                f"mocap-cvs={len(kinect_mocap_csv_list_ok)} vs mocap-xdf={len(kinect_mocap_xdf_list_ok)}"
            )

    if not error_msg:

        kinect_mocap_xdf_timestamps = [x[0] for x in kinect_mocap_xdf_list_ok]
        kinect_mocap_csv_timestamps = [x[0] for x in kinect_mocap_csv_list_ok]
        # get the differences between the timestamps
        kinect_mocap_timestamps_diff = [
            kinect_mocap_xdf_timestamps[i] - kinect_mocap_csv_timestamps[i]
            for i in range(len(kinect_mocap_csv_timestamps))
        ]

        # get the mean and the standard deviation of the differences
        kinect_mocap_timestamps_diff_mean = np.mean(kinect_mocap_timestamps_diff)
        kinect_mocap_timestamps_diff_std = np.std(kinect_mocap_timestamps_diff)

        warning_msg += "\n" + (
            f"\nkinect_mocap_timestamps_diff_mean: {kinect_mocap_timestamps_diff_mean:.3f}ss"
        )
        warning_msg += "\n" + (
            f"    Standard deviation: {kinect_mocap_timestamps_diff_std:.6f} s"
        )

        warning_msg += "\n" + (
            f"\nkinect_markers_timestamps_diff_mean: {np.mean(k_delays['timestamps_differences']):.3f} s"
        )
        warning_msg += "\n" + (
            f"    Standard deviation: {np.std(k_delays['timestamps_differences']):.6f} s"
        )

        marker_mocap_timestamps_diff_mean = (
            np.mean(k_delays["timestamps_differences"])
            - kinect_mocap_timestamps_diff_mean
        )
        warning_msg += "\n" + (
            f"\nDifference between marker and mocap delays: {marker_mocap_timestamps_diff_mean:.6f} s"
        )
        if marker_mocap_timestamps_diff_mean < 0.001:
            warning_msg += "\n" + ("The difference is less than 1 ms :-)")
        else:
            warning_msg += "\n" + (
                "WARNING: The difference is greater than 1 ms, although smaller than 100 ms :-( "
            )
        if marker_mocap_timestamps_diff_mean > 0.1:
            warning_msg += "\n" + "The difference is greater than 100 ms\n"

    return {
        "kinect_mocap_timestamps_diff_mean": kinect_mocap_timestamps_diff_mean,
        "kinect_mocap_timestamps_diff_std": kinect_mocap_timestamps_diff_std,
        "error_msg": error_msg,
        "warning_msg": warning_msg,
    }


def print_verify_kinect_to_csv_mocap_delay(verify_result):
    """Print the verification result of the kinect to csv delay by comparing the kinect mocap in xdf and csv files"""

    kinect_mocap_timestamps_diff_mean = verify_result[
        "kinect_mocap_timestamps_diff_mean"
    ]
    kinect_mocap_timestamps_diff_std = verify_result["kinect_mocap_timestamps_diff_std"]
    error_msg = verify_result["error_msg"]
    warning_msg = verify_result["warning_msg"]

    print("")
    print(
        "******BEGIN: Verification of kinect to csv delay comparing the kinect mocap in xdf and csv files:"
    )
    print(f"Kinect mocap timestamps diff mean: {kinect_mocap_timestamps_diff_mean:.3f}")
    print(f"Kinect mocap timestamps diff std : {kinect_mocap_timestamps_diff_std:.6f}")
    print("--> Warnings: " + warning_msg)
    print("--> Error: " + error_msg)
    print(
        f"********END: Verification of kinect to csv delay comparing the kinect mocap in xdf and csv files"
    )
    return


if doRunTests:
    xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(xdf_path)
    for xdf_csv in xdf_and_csv_files_in_visit:
        xdf_rel_fname = xdf_csv["xdf"]
        csv_rel_fnames = xdf_csv["csv"]
        mandatory_files_error = xdf_csv["mandatory_files_error"]
        expected_files_warning = xdf_csv["expected_files_warning"]

        print(
            "\n*****************************************************************************"
            + "\n*****************************************************************************"
        )
        print(f"\nXDF: {xdf_rel_fname}")

        if mandatory_files_error:
            print(f"  >>>>>> ERROR: {mandatory_files_error}")
        else:
            # do the analysis
            verify_result = verify_kinect_to_csv_mocap_delay(
                xdf_rel_fname, csv_rel_fnames
            )
            print_verify_kinect_to_csv_mocap_delay(verify_result)

In [ ]:
def check_csv_date(fullFname, msg=""):
    """
    Check the date of a csv file
    The date is extracted from the first timestamp in the file
    """
    # date is set to 1970-01-01 00:00:00 if not found
    # date = datetime.fromtimestamp(0, tz=None)
    # date is set to None if not found
    date = None

    if not fullFname.endswith(".csv"):
        msg += f"{fullFname} is not a csv file."
        return date, msg

    # The timestamps are always in column 1, after 3-4 lines of header
    # but it can be a string or a float (in milliseconds)
    data = np.loadtxt(fullFname, skiprows=4, delimiter=",", max_rows=5, dtype=str)

    # check if the file is empty
    if data.size == 0:
        basename = os.path.basename(fullFname)
        msg += f"{basename} is an empty file."
        return date, msg
    # check if we get the expected number of columns
    if data.shape[1] < 2:
        basename = os.path.basename(fullFname)
        msg += f"{basename} has less than 2 columns."
        return date, msg

    # Sounds good, we have some data to check
    # we only need the first timestamp
    timestamp = data[0, 0]
    try:
        timestamp = float(timestamp) / 1000  # in seconds
    except:
        pass
    if isinstance(timestamp, float):
        date = datetime.fromtimestamp(timestamp, tz=None)
    if isinstance(timestamp, str):
        date = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S.%f")
    return date, msg


def sort_csv_files_by_date(csv_files):
    """Sort the csv files by date inside the file"""

    csv_files_sorted = []
    for csv_file in csv_files:
        csv_full_fname = os.path.join(xdf_path, csv_file)
        date, msg = check_csv_date(csv_full_fname)
        if date is not None:
            csv_files_sorted.append([date, csv_file, msg])

    csv_files_sorted.sort(key=lambda x: x[0])

    # get the time difference between the first file and the other files
    t_zero_sec = csv_files_sorted[0][0].timestamp()
    diff_time = []
    diff_time.append(t_zero_sec - t_zero_sec)
    for csv_file in csv_files_sorted:
        timestamp_sec = csv_file[0].timestamp()
        diff_time = np.append(diff_time, timestamp_sec - t_zero_sec)

    for i in range(len(csv_files_sorted)):
        csv_files_sorted[i].append(diff_time[i])

    return csv_files_sorted


def print_csv_files_by_date(csv_files_sorted):
    """Print the csv files sorted by date"""

    # we need this local import to avoid conflict with global import:
    # from datetime import datetime (for the function check_csv_date)

    from datetime import timedelta

    print("\nCSV files sorted by date (date inside the file):")
    for csv_file in csv_files_sorted:
        date = csv_file[0]
        csv_fname = csv_file[1]
        msg = csv_file[2]
        diff_time = csv_file[3]
        diff_time_HMS = timedelta(seconds=diff_time)
        diff_time_HMS_str = str(diff_time_HMS)
        # remove the last 3 characters (milliseconds)
        diff_time_HMS_str = diff_time_HMS_str[:-3]
        # print(f"{date}: ({diff_time:11.3f}) {csv_fname} {msg}")
        print(f"{date}: (+{diff_time_HMS_str}) {csv_fname} {msg}")

    return

## Compute the kinect-to-mouse delay for one xdf file

In [ ]:
def get_kinect_to_mouse_delay_for_xdf_file(xdf_rel_fname, csv_rel_fnames):
    """Get the kinect to mouse delay for an xdf file and the corresponding csv files"""

    ktm_delays = get_kinect_to_mouse_markers_delay_for_one_xdf_dict(
        xdf_rel_fname, csv_rel_fnames
    )

    if ktm_delays["error_msg"]:
        return {
            "error_msg": ktm_delays["error_msg"],
            "warning_msg": "",
            "ktm_delays": [],
        }

    # verify the kinect to csv delay by comparing the kinect mocap in xdf and csv files
    verify_result = verify_kinect_to_csv_mocap_delay(xdf_rel_fname, csv_rel_fnames)

    return {
        "error_msg": verify_result["error_msg"],
        "warning_msg": verify_result["warning_msg"],
        "ktm_delays": ktm_delays,
    }


if doRunTests:
    xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(xdf_path)
    for xdf_csv in xdf_and_csv_files_in_visit:
        xdf_rel_fname = xdf_csv["xdf"]
        csv_rel_fnames = xdf_csv["csv"]
        mandatory_files_error = xdf_csv["mandatory_files_error"]
        expected_files_warning = xdf_csv["expected_files_warning"]

        print(
            "\n*****************************************************************************"
            + "\n*****************************************************************************"
        )
        print(f"\nXDF: {xdf_file}")

        if mandatory_files_error:
            print(f"  >>>>>> ERROR: {mandatory_files_error}")
        else:
            # do the analysis
            for csv_fname in csv_rel_fnames:
                print(f"csv: {csv_fname}")
            print(f"\nMandatory files error: {mandatory_files_error}")
            print(f"\nExpected files warning: {expected_files_warning}")

            if not mandatory_files_error:  # if there is no error
                # get the kinect to mouse delay
                ktm_delays = get_kinect_to_mouse_markers_delay_for_one_xdf_dict(
                    xdf_rel_fname, csv_rel_fnames
                )
                print_kinect_to_mouse_delay_dict(ktm_delays)
                # verify_kinect_to_csv_delay(xdf_rel_fname, csv_rel_fnames)


# ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf and 6 csv files -21495.123 s --- rkm 2021-03-22 14:58:53
# ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf and 6 csv files : -22851.839 s --- ckm 2021-03-22 15:18:03

In [ ]:
if doRunTests:
    # get the whole list of csv files
    csv_rel_fnames_all = []
    xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(xdf_path)
    for xdf_csv in xdf_and_csv_files_in_visit:
        xdf_rel_fname = xdf_csv["xdf"]
        csv_rel_fnames = xdf_csv["csv"]
        mandatory_files_error = xdf_csv["mandatory_files_error"]
        expected_files_warning = xdf_csv["expected_files_warning"]
        csv_rel_fnames_all.extend(csv_rel_fnames)

    # sort the csv files by date
    csv_files_sorted = sort_csv_files_by_date(csv_rel_fnames_all)
    print_csv_files_by_date(csv_files_sorted)

In [ ]:
Expected_order = [
    "_r_k_m.csv",
    "_r_k.csv",
    "_r_l_m_sau_p.csv",
    "_r_l_m_sau_np.csv",
    "_r_l_m_mau_p.csv",
    "_r_l_m_mau_np.csv",
    "_c_k_m.csv",
    "_c_k.csv",
    "_c_l_m_p.csv",
    "_c_l_p.csv",
    "_c_l_m_np.csv",
    "_c_l_np.csv",
]


def is_in_expected_order(csv_files_sorted, expected_order):
    """Check if the csv files are in the expected order"""

    csv_files_sorted_names = [csv_file[1] for csv_file in csv_files_sorted]
    for i, csv_file in enumerate(csv_files_sorted_names):
        # if the csv files does not ends with the expected order, return False
        if not csv_file.endswith(expected_order[i]):
            return False
    return True


if doRunTests:
    if is_in_expected_order(csv_files_sorted, Expected_order):
        print("The csv files are in the expected order")
    else:
        print("The csv files are NOT in the expected order: ")

        csv_files_sorted_names = [csv_file[1] for csv_file in csv_files_sorted]
        for i, csv_file in enumerate(csv_files_sorted_names):
            print(
                f"{i}: {csv_file} -- {csv_file.endswith(Expected_order[i])} -- {Expected_order[i]}"
            )

## Compute the kinect-to-mouse delay for all xdf files in the visit directory



In [ ]:
def get_kinect_to_mouse_delay_for_all_xdf_in_visit(visit_path):
    """Compute the kinect-to-mouse delay for all xdf files in the visit directory"""

    xdf_delays = []

    xdf_and_csv_files_in_visit = get_xdf_and_csv_files_in_visit_by_xdf(visit_path)
    for xdf_csv in xdf_and_csv_files_in_visit:
        xdf_rel_fname = xdf_csv["xdf"]
        csv_rel_fnames = xdf_csv["csv"]
        mandatory_files_error = xdf_csv["mandatory_files_error"]
        expected_files_warning = xdf_csv["expected_files_warning"]

        if mandatory_files_error:
            xdf_delays.append(
                {
                    "xdf": xdf_rel_fname,
                    "kinect_to_mouse_delay_mean": np.nan,
                    "kinect_to_mouse_delay_std": np.nan,
                    "kinect_to_mouse_delay_list": [],
                    "mouse_to_csv_delay_list": [],
                    "kinect_to_csv_mocap_delay": np.nan,
                    "mandatory_files_error": mandatory_files_error,
                    "expected_files_warning": expected_files_warning,
                }
            )
        else:
            # do the analysis
            ktm_delays = get_kinect_to_mouse_markers_delay_for_one_xdf_dict(
                xdf_rel_fname, csv_rel_fnames
            )
            # print_kinect_to_mouse_delay_dict(ktm_delays)
            kinect_to_csv_mocap_delay = verify_kinect_to_csv_mocap_delay(
                xdf_rel_fname, csv_rel_fnames
            )
            # print_verify_kinect_to_csv_mocap_delay(verify)

            if not ktm_delays["error_msg"]:
                xdf_delays.append(
                    {
                        "xdf": xdf_rel_fname,
                        "kinect_to_mouse_delay_mean": ktm_delays[
                            "kinect_to_mouse_delay_mean"
                        ],
                        "kinect_to_mouse_delay_std": ktm_delays[
                            "kinect_to_mouse_delay_std"
                        ],
                        "kinect_to_mouse_delay_list": ktm_delays[
                            "kinect_to_cvs_delay_list"
                        ],
                        "mouse_to_csv_delay_list": ktm_delays[
                            "mouse_to_csv_delay_list"
                        ],
                        "kinect_to_csv_mocap_delay": kinect_to_csv_mocap_delay,
                        "mandatory_files_error": mandatory_files_error,
                        "expected_files_warning": expected_files_warning,
                    }
                )
    return xdf_delays


if doRunTests:

    kinect_to_mouse_delay_for_all_xdf_in_visit = (
        get_kinect_to_mouse_delay_for_all_xdf_in_visit(xdf_path)
    )
    for kinect_to_mouse_delay in kinect_to_mouse_delay_for_all_xdf_in_visit:
        print(f"{kinect_to_mouse_delay['xdf']}:")
        if kinect_to_mouse_delay["mandatory_files_error"]:
            print(f"  >>>>>> ERROR: {kinect_to_mouse_delay['mandatory_files_error']}")
        else:
            print(
                f"  kinect_to_mouse_delay_mean: {kinect_to_mouse_delay['kinect_to_mouse_delay_mean']:.3f} s"
            )
            print(
                f"  kinect_to_mouse_delay_std : {kinect_to_mouse_delay['kinect_to_mouse_delay_std']:.6f} s"
            )
            print(
                "**--**--****--**--****--**--****--**--****--**--****--**--****--**--****--**--****--**--****--**--**"
            )